# TP1 — Retrieval : de la question aux passages pertinents

**Objectif** : construire la première moitié d'un RAG, la **recherche** (*retrieval*).
À la fin de ce TP, à partir d'une question en langage naturel, on saura retrouver les passages
des datasheets qui contiennent la réponse — sans encore utiliser de LLM.

```
   INDEXATION (une fois)          PDF ──► pages ──► chunks ──► embeddings ──► index
                                                                                 │
   RECHERCHE (à chaque question)  question ──► embedding ──► similarité ─────────┘──► top-k chunks
```

Plan :
1. Les tokens
2. Les embeddings
3. Du PDF aux chunks
4. Index et recherche sémantique
5. Recherche par mots-clés (BM25) et recherche hybride — *exercices*

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

import settings

load_dotenv()  
warnings.filterwarnings("ignore", message=".*mean pooling.*")  # avertissement interne de fastembed, sans conséquence

## 1. Les tokens

Un modèle de langage ne lit pas des mots mais des **tokens** : des morceaux de texte issus d'un
vocabulaire fixe (quelques dizaines à centaines de milliers d'entrées).

Pourquoi c'est important :
- les appels aux LLM sont **facturés au token** 
- la **fenêtre de contexte** (la quantité de texte qu'on peut donner au modèle) se compte en tokens

Le **tokenizer** est le programme qui fait ce découpage. Il est construit une fois pour toutes à partir d'un corpus, et chaque modèle a le
sien : taille du vocabulaire, langues du corpus et méthode de découpage diffèrent. Le même texte ne donne donc pas le même nombre de tokens d'un modèle à l'autre.

Comparons trois tokenizers (ouverts, téléchargés depuis Hugging Face) sur la même phrase en français et en anglais.

In [ ]:
from tokenizers import Tokenizer

phrase_fr = "Le capteur HC-SR04 mesure des distances de 2 cm à 400 cm."
phrase_en = "The HC-SR04 sensor measures distances from 2 cm to 400 cm."

for nom in ["openai-community/gpt2", "google-bert/bert-base-uncased", "Qwen/Qwen2.5-7B-Instruct"]:
    tokenizer = Tokenizer.from_pretrained(nom)
    print(f"--- {nom} (vocabulaire : {tokenizer.get_vocab_size()} tokens)")
    for phrase in (phrase_fr, phrase_en):
        tokens = tokenizer.encode(phrase).tokens
        print(f"{len(tokens):>3} tokens : {tokens}")
    print()

**À observer**

- Le français coûte plus de tokens que l'anglais avec les trois tokenizers.
- Les références techniques (`HC-SR04`) sont éclatées en 4 ou 5 tokens.
- Les caractères spéciaux (`Ġ`, `Ãł`) viennent du découpage au niveau des *octets* : le modèle ne voit jamais de lettres.

Une datasheet de 60 pages représente plusieurs dizaines de milliers de tokens : on ne peut pas tout donner au LLM à
chaque question. **Il faut sélectionner les bons passages**, c'est l'objet du retrieval.

## 2. Les embeddings

Un **modèle d'embeddings** transforme un texte en un vecteur de nombres (quelques centaines de dimensions), construit
pour que deux textes de sens proche donnent des vecteurs proches. On compare deux vecteurs avec la **similarité cosinus**
(1 = même direction, 0 = sans rapport).

On utilise des modèles ouverts de Hugging Face, petits (20 à 120 millions de paramètres) et exécutés **en local sur le CPU**
grâce à la librairie `fastembed`. Le modèle est téléchargé au premier appel.

In [ ]:
from fastembed import TextEmbedding

phrases = [
    "Le capteur mesure la distance avec des ultrasons.",       # 0 : français
    "The sensor measures distance using ultrasound.",          # 1 : la même phrase en anglais
    "An ultrasonic module for non-contact range finding.",     # 2 : même idée, autres mots
    "The board is powered through the USB connector.",         # 3 : autre sujet technique
    "Il pleut à Angers cet après-midi.",                       # 4 : sans rapport
]

modele = TextEmbedding(settings.EMBEDDING_MODEL)
vecteurs = np.array(list(modele.embed(phrases)))

print("matrice :", vecteurs.shape, "(phrases x dimensions)")
print("début du vecteur de la phrase 0 :", vecteurs[0][:6].round(3), "...")

In [ ]:
def cosinus(a, b):
    """Similarité cosinus entre deux vecteurs."""
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

similarites = np.array([[cosinus(a, b) for b in vecteurs] for a in vecteurs])

plt.imshow(similarites, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="similarité cosinus")
plt.xticks(range(len(phrases)))
plt.yticks(range(len(phrases)), [f"{i} : {p[:38]}…" for i, p in enumerate(phrases)])
for i in range(len(phrases)):
    for j in range(len(phrases)):
        plt.text(j, i, f"{similarites[i, j]:.2f}", ha="center", va="center",
                 color="white" if similarites[i, j] > 0.6 else "black")
plt.title("Similarité entre les phrases")
plt.show()

**À observer**

- 0 ↔ 1 (même phrase, deux langues) et 1 ↔ 2 (même idée, mots différents) sont proches, **sans avoir aucun mot en commun** :
  le vecteur capture le *sens*, pas les mots.
- La phrase 3 est "un peu" proche des autres (vocabulaire technique commun).
- La phrase 4 est loin de tout.

### 2.b Tous les modèles ne se valent pas

Il existe des centaines de modèles d'embeddings. Ils diffèrent par leur taille, leur **nombre de dimensions**,
les **langues** vues à l'entraînement et leur qualité. Comparons trois modèles sur nos phrases.

In [ ]:
modeles = settings.EMBEDDING_MODELS_TO_COMPARE  # la liste est dans settings.py

print(f"{'modèle':<62} {'dim':>4}  FR↔EN  EN↔EN'  EN↔pluie")
for nom in modeles:
    v = np.array(list(TextEmbedding(nom).embed(phrases)))
    print(f"{nom:<62} {v.shape[1]:>4}  {cosinus(v[0], v[1]):5.2f}  {cosinus(v[1], v[2]):5.2f}   {cosinus(v[1], v[4]):5.2f}")

**À observer**

- Les modèles n'ont pas le même nombre de dimensions (384, 768…) : leurs vecteurs ne sont **pas comparables entre eux**.
  Si on change de modèle, il faut **recalculer tout l'index**.
- `all-MiniLM-L6-v2` (anglais) ne voit presque aucun lien entre la phrase française et sa traduction (FR↔EN ≈ 0,3) :
  pour poser des questions en français sur des documents en anglais, il faut un modèle **multilingue** (FR↔EN ≈ 0,9).
- `bge-base-en` donne FR↔EN ≈ 0,7… mais aussi ≈ 0,5 entre deux phrases sans aucun rapport. Chaque modèle a **sa propre
  échelle** de similarité : une valeur de cosinus n'a de sens que *relativement* aux autres, il n'y a pas de seuil universel.
- Le choix du modèle est un compromis qualité / taille / vitesse / langues.

## 3. Du PDF aux chunks

Notre corpus : 6 datasheets (Raspberry Pi Pico, ESP32, Arduino Uno, BME280, HC-SR04, LM35), ~240 pages.

On ne calcule pas un vecteur par document : un seul vecteur pour 60 pages "dilue" le sens et ne permet pas de
retrouver *le* passage qui répond à la question posée. On découpe donc les documents en **chunks** (morceaux) de quelques dizaines à
quelques centaines de mots, et c'est chaque chunk qui sera vectorisé.

Étape 1 : lire les PDF (`load_pdfs` dans `rag.py`). On garde pour chaque page sa source et son numéro : ils serviront à citer.

In [ ]:
from collections import Counter
from rag import load_pdfs, chunk_pages

pages = load_pdfs()
print(Counter(page["source"] for page in pages))

page = [p for p in pages if p["source"] == "hcsr04.pdf"][0]
print("\n--- texte brut de la page 1 du HC-SR04 ---")
print(page["text"][:700])

Le texte extrait d'un PDF est **sale** : tableaux aplatis, en-têtes et pieds de page répétés, sauts de ligne arbitraires,
parfois plusieurs langues (ex : datasheet Arduino). La qualité du RAG dépend beaucoup de cette étape,
souvent sous-estimée.

Étape 2 : découper. `chunk_pages` coupe chaque page tous les 150 mots avec 30 mots de chevauchement
(`settings.CHUNK_SIZE_WORDS`, `settings.CHUNK_OVERLAP_WORDS`). C'est la méthode la plus naïve possible.

In [ ]:
chunks = chunk_pages(pages)
print(len(chunks), "chunks")
print()
print(chunks[400])

Un chunk = un dictionnaire `{"id", "source", "page", "text"}`. Le chevauchement évite qu'une information
soit coupée en deux juste à la frontière entre deux chunks.

## 4. Index et recherche sémantique

Étapes 3 et 4 : vectoriser tous les chunks et sauvegarder le résultat dans `index/` (`chunks.json` + `embeddings.npy`).
Sur CPU cela prend **2 à 3 minutes** : profitez-en pour lire `build_index` et `search` dans `rag.py`.

In [ ]:
from rag import build_index, load_index

build_index()
chunks, embeddings = load_index()
print("\nmatrice des embeddings :", embeddings.shape, "(chunks x dimensions)")

La recherche sémantique tient en trois lignes de numpy :
1. vectoriser la question **avec le même modèle** que les chunks
2. calculer la similarité avec tous les chunks d'un coup (un produit matrice × vecteur)
3. garder les k plus grands scores

Chaque chunk est maintenant un point dans un espace à 384 dimensions, et la question en est un autre : chercher,
c'est trouver les points les plus proches.

In [ ]:
from rag import embed

question = "What is the operating voltage of the HC-SR04?"

vecteur_question = embed([question])[0]                 # 1. (les vecteurs sont normalisés : cosinus = produit scalaire)
scores = embeddings @ vecteur_question                   # 2. un score par chunk
meilleurs = np.argsort(scores)[::-1][:5]                 # 3. indices des 5 plus grands scores

for i in meilleurs:
    print(f"{scores[i]:.3f}  {chunks[i]['source']:<16} p.{chunks[i]['page']:<3} {chunks[i]['text'][:90]!r}")

C'est exactement ce que fait `search` dans `rag.py`. Testons sur d'autres questions, en français.

In [ ]:
from rag import search

def afficher(resultats):
    for r in resultats:
        print(f"{r['score']:8.3f}  {r['source']:<16} p.{r['page']:<3} {r['text'][:90]!r}")

questions = [
    "Quelle est la fréquence maximale du processeur de l'ESP32 ?",
    "Quelle est la tension d'alimentation du HC-SR04 ?",
    "À quelle adresse se trouve le registre ctrl_meas du BME280 ?",
]
for question in questions:
    print("\n###", question)
    afficher(search(question, chunks, embeddings, k=5))

**À observer**

- Souvent, ça marche : la bonne datasheet et la bonne page remontent (ESP32).
- Parfois, le bon document n'est même pas dans le top 5 (HC-SR04 : un document de 3 pages, noyé par des chunks
  "alimentation / tension" des autres datasheets).
- Sur un identifiant exact (`ctrl_meas`, un nom de registre), la recherche sémantique passe à côté : les pages qui
  décrivent ce registre ne sont pas dans le top 5. Un vecteur de 384 nombres résume le *sens* d'un chunk, pas les *mots*
  qu'il contient.

## 5. Recherche par mots-clés et recherche hybride

**BM25** est l'algorithme des moteurs de recherche classiques : le score d'un chunk dépend du nombre de mots
de la question qu'il contient, pondérés par leur rareté dans le corpus (un mot présent partout ne compte presque pas).

`search_bm25` dans `rag.py` utilise la librairie `rank_bm25` et un découpage en mots très simple (`tokenize`).

In [ ]:
from rag import search_bm25

for question in questions:
    print("\n###", question)
    afficher(search_bm25(question, chunks, k=5))

**À observer**

- Les trois pages qui décrivent `ctrl_meas` arrivent en tête, et le HC-SR04 remonte dans les premiers :
  BM25 excelle sur les **termes exacts**.
- Mais pour l'ESP32, aucun chunk pertinent : BM25 remonte les pages d'avertissements réglementaires (en français !)
  de la datasheet Arduino, parce que les mots `de`, `la`, `est`, `du` comptent autant que les autres.
  BM25 ignore les synonymes et ne connaît pas la langue.

Les deux approches ont des forces complémentaires. En pratique, on les **combine** : c'est la recherche **hybride**.

### Exercice 1 — recherche hybride (en deux étapes)

Tout se fait dans les deux cellules ci-dessous, qui contiennent les squelettes à compléter ; `rag.py` reste inchangé.

**a) Les mots vides.** Complétez `tokenize_sans_mots_vides` : comme `rag.tokenize`, mais en ignorant les mots de
`MOTS_VIDES` (une liste de départ est fournie). `search_bm25_sans_mots_vides`, déjà écrite, l'utilise.
Relancez-la sur la question française : la page Arduino doit disparaître.

**b) La fusion.** Complétez `search_hybrid`, qui combine les résultats de `search` et de `search_bm25_sans_mots_vides`.
Méthode conseillée : **Reciprocal Rank Fusion (RRF)**. Pour chaque chunk, on additionne, sur les deux listes,
`1 / (60 + rang)` (rang à partir de 1 ; un chunk absent d'une liste n'apporte rien). Un chunk bien classé
dans les deux listes remonte en tête. Les scores bruts ne sont pas comparables (cosinus entre -1 et 1, BM25 non borné) :
c'est pour cela qu'on fusionne des *rangs* et non des *scores*.
Variante : somme pondérée des scores après normalisation (min-max) de chaque liste.

Testez sur les trois questions ci-dessus.

In [ ]:
import re
from rank_bm25 import BM25Okapi

MOTS_VIDES = set("""
le la les l un une des du de d et ou à au aux en dans par pour sur avec sans est sont a ont que qui quoi
quel quelle quels quelles comment combien ce cet cette ces se sa son ses il elle on ne pas plus
the a an of to in on for and or is are be by with at from as that this which what how many much it its
""".split())


def tokenize_sans_mots_vides(texte):
    # a) À compléter : comme rag.tokenize, mais en ignorant les mots de MOTS_VIDES
    ...


def search_bm25_sans_mots_vides(question, chunks, k=5):
    # Même chose que rag.search_bm25, avec le nouveau découpage en mots
    bm25 = BM25Okapi([tokenize_sans_mots_vides(c["text"]) for c in chunks])
    scores = bm25.get_scores(tokenize_sans_mots_vides(question))
    meilleurs = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in meilleurs]


# à décommenter une fois la fonction complétée :
# afficher(search_bm25_sans_mots_vides("Quelle est la tension d'alimentation du HC-SR04 ?", chunks))

In [ ]:
def search_hybrid(question, chunks, embeddings, k=5, n=20):
    # b) À compléter : combine recherche sémantique et BM25 par fusion de rangs (RRF)
    resultats_semantique = search(question, chunks, embeddings, k=n)
    resultats_bm25 = search_bm25_sans_mots_vides(question, chunks, k=n)

    # À compléter : calculer un score fusionné par chunk (clé : chunk["id"]),
    # puis renvoyer les k chunks aux meilleurs scores, avec leur score dans "score".
    ...


# à décommenter une fois la fonction complétée :
# for question in questions:
#     print("\n###", question)
#     afficher(search_hybrid(question, chunks, embeddings, k=5))

### Exercice 2 — améliorer le découpage

`chunk_pages` coupe tous les 150 mots, en plein milieu des phrases et des tableaux. Améliorez-la, par exemple :

- couper sur les paragraphes (`"\n\n"`) ou les phrases plutôt qu'au mot
- changer la taille (50 ? 300 mots ?) et le chevauchement
- supprimer les en-têtes / pieds de page répétés
- ajouter le nom du document au début de chaque chunk 
- etc.

Tout se fait dans les cellules ci-dessous ; `rag.py` et l'index de référence (`index/`) restent inchangés, on les garde
pour comparer. Complétez `chunk_pages_par_paragraphes`, puis exécutez la cellule suivante : elle calcule un **second index** (2 à 3 min),
le sauvegarde dans `index_paragraphes/` (pour le réutiliser au TP3 sans recalculer) et rejoue les mêmes questions — `search` fonctionne avec n'importe quel couple (chunks, embeddings).
Le TP3 permettra de **mesurer** objectivement quel découpage est le meilleur.


In [ ]:
def chunk_pages_par_paragraphes(pages, max_words=150):
    # À compléter : découper chaque page sur les paragraphes ("\n\n") en regroupant les paragraphes
    # consécutifs jusqu'à max_words mots ; chaque chunk garde id, source, page, text (voir rag.chunk_pages).
    # Idée bonus : commencer chaque texte par le nom du document.
    chunks_resultat = []
    for page in pages:
        paragraphes = [p for p in page["text"].split("\n\n") if p.strip()]
        ...
    return chunks_resultat


In [ ]:
# à décommenter une fois la fonction complétée :
# chunks_paragraphes = chunk_pages_par_paragraphes(pages)
# embeddings_paragraphes = embed([c["text"] for c in chunks_paragraphes])
# print(len(chunks_paragraphes), "chunks (contre", len(chunks), "avec le découpage de base)")
#
# for question in questions:
#     print("\n###", question)
#     afficher(search(question, chunks_paragraphes, embeddings_paragraphes, k=5))
#
# # on sauvegarde ce second index : le TP3 le rechargera sans recalculer les embeddings
# from rag import save_index
# save_index(chunks_paragraphes, embeddings_paragraphes, "index_paragraphes")

## Bilan

On a construit le **R** de RAG :

```
load_pdfs ──► chunk_pages ──► embed ──► build_index          (rag.py, étapes 1 à 4)
question ──► search / search_bm25 / search_hybrid ──► top-k   (rag.py, étape 5)
```

Cet après-midi (TP2), ces k chunks seront donnés à un LLM pour **générer** la réponse.

*Dans un système en production, la matrice numpy est remplacée par une base de données vectorielle (pgvector, Qdrant, LanceDB, …)
qui fait la même chose à grande échelle. Le principe ne change pas.*

*Pas eu le temps de finir les exercices ? Le corrigé (`solutions/tp1.py`) est fourni en fin de TP1.
Le TP2 et le TP3 importent ses fonctions : vous pourrez continuer sans être bloqué.*